# 06 - Sensitivity and reporting

Every estimate so far has rested on conditional ignorability: the assumption
that we measured everything driving both treatment and outcome. It is
untestable, and it is almost never exactly true.

Sensitivity analysis replaces the unanswerable question "is it true?" with an
answerable one: **how wrong would it have to be to change the conclusion?** If
the answer is "wildly wrong", the finding is robust. If it is "slightly wrong",
the finding was never safe.

This notebook closes the loop — estimate, stress-test, then write the result
down in a form that forces the caveats to travel with the number.

## Causal question

What is the average effect of the treatment on the outcome, and how much
unmeasured confounding would be needed to overturn whatever we conclude?

## Data and design

- **Unit of analysis:** one individual.
- **Treatment:** `treatment`, binary, assigned as a function of covariates.
- **Outcome:** `outcome`, continuous.
- **Covariates:** `x1`, `x2`, `x3`, all pre-treatment.

The generator's true ATE is known, which lets us check the estimate — and, more
usefully here, check that the sensitivity machinery behaves as advertised.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

import numpy as np
import pandas as pd

from causal_inference_lab.data_generators import make_confounded_binary_treatment
from causal_inference_lab.estimators import aipw_ate, difference_in_means
from causal_inference_lab.reporting import CausalReport
from causal_inference_lab.sensitivity import omitted_confounder_simulation, placebo_treatment_test
from causal_inference_lab.uncertainty import bootstrap_ate

COVARIATES = ["x1", "x2", "x3"]

dataset = make_confounded_binary_treatment(n=5_000, seed=42)
data = dataset.data

print(f"observations:  {len(data):,}")
print(f"treated share: {data['treatment'].mean():.1%}")
print(f"true ATE:      {dataset.true_ate:.3f}")

**Interpretation.** A conventional setup. Nothing about the data announces
whether an unmeasured confounder exists — which is the entire difficulty, and
the reason the rest of this notebook is necessary.

## Estimand

The **average treatment effect (ATE)**.

## Identification assumptions

1. **Consistency.** The observed outcome is the potential outcome under the
   treatment received.
2. **No interference.** One unit's treatment does not affect another's outcome.
3. **Conditional ignorability** given `x1`, `x2`, `x3`. Untestable, and the
   target of everything below.
4. **Overlap.** Both treatment arms are possible for every covariate profile.

Assumptions 1 and 2 are design facts. Assumption 4 is checkable. Assumption 3 is
neither, so we quantify our exposure to its failure instead.

## Estimation

AIPW, with the naive contrast alongside it to show how much work adjustment is
doing.

In [ ]:
naive = difference_in_means(data)
estimate = aipw_ate(data, covariates=COVARIATES)

print(f"naive difference in means: {naive.estimate:.3f}")
print(f"AIPW:                      {estimate.estimate:.3f}")
print(f"true ATE:                  {dataset.true_ate:.3f}")
print(f"\nconfounding removed by adjustment: {naive.estimate - estimate.estimate:.3f}")

**Interpretation.** Adjustment moves the estimate by 1.76, from 3.80 to 2.04
against a true 1.99. The measured covariates were doing a great deal of work.
That is precisely why the possibility of an *unmeasured* one is worth taking
seriously: confounding of this magnitude is clearly present in this setting, and
nothing guarantees we caught all of it.

## Diagnostics

A placebo test asks whether the estimator finds an effect where none can exist.
Shuffling the treatment labels destroys any real relationship while leaving the
data's structure intact, so an estimator behaving correctly should return
approximately zero.

In [ ]:
placebo = placebo_treatment_test(
    data=data,
    estimator=aipw_ate,
    covariates=COVARIATES,
    seed=123,
)

print(f"test:               {placebo.name}")
print(f"real estimate:      {placebo.reference_estimate:.4f}")
print(f"placebo estimate:   {placebo.estimate:.4f}")
print(f"ratio to real:      {abs(placebo.estimate) / abs(placebo.reference_estimate):.4f}")
print(f"\n{placebo.interpretation}")

**Interpretation.** The placebo estimate is 0.0015 against a real estimate of
2.04 — smaller by a factor of about 1,400. The estimator is not manufacturing
effects out of the covariate structure, and there is no sign of leakage between
treatment and outcome.

Be clear about what this rules out. A passing placebo test says the *machinery*
is sound. It says nothing whatever about unmeasured confounding, because
shuffled labels are unconfounded by construction. A real confounder would leave
this test entirely unmoved.

That is what the omitted-confounder simulation is for. It posits a confounder of
a given strength, works out the bias such a confounder would induce, and reports
what the estimate would become.

In [ ]:
sensitivity = omitted_confounder_simulation(
    data=data,
    base_effect=estimate.estimate,
    confounder_strength_grid=[0.0, 0.25, 0.5, 0.75, 1.0, 1.5],
)
print(sensitivity.to_string(index=False, float_format=lambda v: f"{v:.3f}"))

flips_sign = sensitivity.loc[sensitivity["adjusted_effect"] <= 0]
print(f"\nstrength at which the effect would vanish: "
      f"{'none in this grid' if flips_sign.empty else flips_sign.iloc[0]['confounder_strength']}")

**Interpretation.** The bias grows with the square of the assumed strength, so
the estimate erodes slowly at first and then quickly: 2.04 at strength zero,
1.79 at 0.5, 1.04 at 1.0, and only at 1.5 does it turn negative.

The reading is that the *sign* of this finding is robust — a confounder would
have to be extremely strong, comparable in influence to the treatment itself, to
overturn the direction. The *magnitude* is not robust: a moderate confounder at
strength 0.75 would cut the effect by 28%. If a decision depends on the effect
exceeding some threshold rather than merely being positive, this table is the
part of the analysis that matters.

## Uncertainty

Sensitivity analysis addresses bias. It says nothing about sampling variability,
which needs its own treatment — bootstrapping the whole AIPW procedure.

In [ ]:
interval = bootstrap_ate(
    data,
    estimator=aipw_ate,
    covariates=COVARIATES,
    n_bootstrap_samples=300,
    seed=42,
)

print(f"estimate:      {interval.estimate:.3f}")
print(f"standard error:{interval.std_error:.3f}")
print(f"95% interval:  [{interval.lower:.3f}, {interval.upper:.3f}]")
print(f"true ATE:      {dataset.true_ate:.3f}")
print(f"covers truth:  {interval.lower <= dataset.true_ate <= interval.upper}")

worst_case = float(sensitivity.loc[sensitivity["confounder_strength"] == 0.75, "adjusted_effect"].iloc[0])
print(f"\nwidth of the 95% interval:            {interval.upper - interval.lower:.3f}")
print(f"shift from a strength-0.75 confounder: {abs(estimate.estimate - worst_case):.3f}")

**Interpretation.** The interval is [1.971, 2.103], width 0.13, and it covers the
truth — though only just: 1.990 sits close to the lower edge.

The comparison at the bottom is the point of running both analyses. Sampling
noise moves the estimate by about 0.13. A plausible unmeasured confounder moves
it by 0.56 — more than four times as much. Reporting the confidence interval
alone would convey a precision the analysis does not have, because the dominant
uncertainty here is not statistical.

Finally the report. `CausalReport` refuses to be constructed without a question,
an estimand, assumptions, diagnostics, uncertainty, limitations, and a
recommendation — so the caveats travel attached to the number rather than being
left behind in a notebook nobody opens.

In [ ]:
report = CausalReport(
    question="What is the average effect of the treatment on the outcome?",
    estimand="ATE",
    identification_assumptions=(
        "Consistency: observed outcomes are potential outcomes under treatment received",
        "No interference between units",
        "Conditional ignorability given x1, x2, x3",
        "Overlap: both arms possible for every covariate profile",
    ),
    estimator="Augmented inverse probability weighting (AIPW)",
    diagnostics={
        "placebo_estimate": round(float(placebo.estimate), 4),
        "placebo_relative_to_real": round(float(abs(placebo.estimate / placebo.reference_estimate)), 4),
        "naive_minus_adjusted": round(float(naive.estimate - estimate.estimate), 3),
    },
    uncertainty={
        "point_estimate": round(float(interval.estimate), 3),
        "ci_95": [round(float(interval.lower), 3), round(float(interval.upper), 3)],
        "bootstrap_samples": interval.n_bootstrap_samples,
        "confounder_strength_0.75_effect": round(worst_case, 3),
    },
    results={
        "ate": round(float(estimate.estimate), 3),
        "naive_ate": round(float(naive.estimate), 3),
        "true_ate_synthetic": round(float(dataset.true_ate), 3),
    },
    limitations=(
        "Conditional ignorability is assumed and cannot be tested",
        "Sensitivity analysis shows the magnitude is not robust to moderate unmeasured confounding",
        "Bias from a plausible confounder exceeds sampling uncertainty by roughly fourfold",
        "Synthetic data: no measurement error, missingness, or model drift",
    ),
    recommendation=(
        "The positive sign is well supported and survives strong assumed confounding. "
        "Do not rely on the point estimate for a threshold decision without first "
        "bounding unmeasured confounding with domain knowledge."
    ),
)

print(report.to_markdown())

**Interpretation.** The recommendation distinguishes what the analysis supports
(a positive effect) from what it does not (a precise magnitude). That
distinction is the deliverable. A report that said "the effect is 2.04, 95% CI
[1.97, 2.10]" would be arithmetically correct and practically misleading,
because it omits the larger source of error.

## Limitations

- **Sensitivity analysis is parametric.** The simulation assumes a particular
  functional form for how a confounder biases the estimate. A confounder acting
  differently produces a different curve.
- **Confounder strength has no natural units.** Reading the table requires
  judgement about what strength is plausible in the application, and that
  judgement is domain knowledge, not statistics.
- **A passing placebo test proves very little.** It detects broken machinery and
  leakage. It cannot detect the failure the whole notebook is about.
- **The bootstrap covers sampling only.** It assumes the model is right, so its
  interval is a lower bound on total uncertainty.
- **`CausalReport` enforces presence, not honesty.** Nothing stops a limitations
  tuple from being complacent; the structure only guarantees the field exists.
- **Synthetic data.** Real applications add measurement error and missingness,
  both of which bias estimates in ways this notebook does not model.